In [1]:
import sys
import os
sys.path.append(os.path.abspath("../.."))
import numpy as np
import pandas as pd
from utilsforecast.preprocessing import fill_gaps
from mlforecast import MLForecast
from mlforecast import MLForecast
from tinyshift.modelling import TwoStageForecasterWrapper
from tinyshift.series import economic_loss, tail_risk
import lightgbm as lgb
from mlforecast.lag_transforms import RollingMean, RollingStd
from tinyshift.series import wape, pbias, forecast_instability


In [2]:
# Gerando dados em painel sintéticos para 5 séries temporais
np.random.seed(42)
n_series = 5
time_steps = 100

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'
df = pd.read_csv(url, parse_dates=['Month'])
df["unique_id"] = "1"
df.rename(columns={"Month": "ds", "Passengers": "y"}, inplace=True)
df = fill_gaps(df, freq="ME", end="per_serie", id_col="unique_id", time_col="ds")
days_obsoletes=180
horizon = 12
train = df[:-horizon]
test = df[-horizon:]

In [3]:
# Financial parameters
selling_price = 320      # Unit selling price
unit_cost = 90         # Unit acquisition cost
annual_holding_rate = 0.14 # Annual capital holding rate (12% p.a.)

days_obsoletes = 180

# 1. Underage Cost (Cu): Lost margin per unfulfilled unit
c_u = selling_price - unit_cost

# 2. Overage Cost (Co): Holding cost + daily obsolescence rate
daily_obsolescence_cost = unit_cost / days_obsoletes
daily_holding_cost = (unit_cost * annual_holding_rate) / 365

estimated_holding_days = 30
c_o = (daily_holding_cost + daily_obsolescence_cost) * estimated_holding_days

In [4]:
fcst = MLForecast(
    models={
        'lgb_issm': lgb.LGBMRegressor(
            objective='poisson',
            metric='rmse',
            n_estimators=100,
            learning_rate=5e-2,
            random_state=42,
            verbosity=-1
        )
    },
    freq='MS',
    lags=[1, 2, 7],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [5]:
tsf = TwoStageForecasterWrapper(fcst)

In [6]:
tsf.fit(train, static_features=[], h=12, n_windows=10, refit=True)

,fcst,MLForecast(mo...num_threads=1)


In [7]:
test = test.copy()
test.loc[:,"cu"] = 2
test.loc[:,"co"] = 1

In [8]:
pred = tsf.optimize(h=12, underage_cost="cu", overage_cost="co", X_df=test)

In [9]:
pred = pd.merge(pred, test, on=["unique_id", "ds"])

In [10]:
economic_loss(pred, ["y_optimal"], id_col="unique_id")

,unique_id,metric,y_optimal
0,1,economic_loss,1098


In [11]:
tail_risk(pred, ["y_optimal"], id_col="unique_id")

,unique_id,metric,y_optimal
0,1,expected_cost,91.50
1,1,std_dev,99.23
2,1,var95,296.00
3,1,cvar95,296.00
4,1,worst_scenario,296.00


In [12]:
pred

,unique_id,ds,lambda_t,r_dispersion,critical_ratio,y_optimal,y,cu,co
0,1,1960-01-01,419.416311,17.525388,0.666667,456,417,2,1
1,1,1960-02-01,438.165120,17.525388,0.666667,477,391,2,1
2,1,1960-03-01,438.165120,17.525388,0.666667,477,419,2,1
3,1,1960-04-01,438.165120,17.525388,0.666667,477,461,2,1
4,1,1960-05-01,438.165120,17.525388,0.666667,477,472,2,1
5,1,1960-06-01,440.933602,17.525388,0.666667,480,535,2,1
6,1,1960-07-01,440.933602,17.525388,0.666667,480,622,2,1
7,1,1960-08-01,420.959324,17.525388,0.666667,458,606,2,1
8,1,1960-09-01,418.916062,17.525388,0.666667,456,508,2,1
9,1,1960-10-01,418.916062,17.525388,0.666667,456,461,2,1


In [13]:
forecast_instability(pred, models=["y_optimal"])

,unique_id,metric,y_optimal
0,1,forecast_instability,0.932039


In [14]:
wape(pred, models=["y_optimal"])

,unique_id,metric,y_optimal
0,1,wape,12.180609


In [15]:
pbias(pred, models=["y_optimal"])

,unique_id,metric,y_optimal
0,1,pbias,-1.890095
